# Day 3b. Memory, and the pause

Two things in one notebook, and the order matters.

**First: memory.** On day 2 you continued a conversation by keeping the message
list in a variable and passing the whole thing back yourself. That works for one
person in one notebook. It does not work for an app.

**Then: the pause.** `book_room` spends money, and so far nothing has stopped
it. We want the run to stop and ask you first. But a run that stops has to be
stored somewhere while it waits, and that store is exactly the thing we build in
part 1. That is why memory comes first.

*GIU AI Connects · Amr Saleh · iHQ Tech*

## 0 · Setup

One `uv` environment for the whole week (see `README.md`): `uv sync`, then run
Jupyter with `uv run jupyter lab`.

Put your API key in a `.env` file next to this notebook:

```
ANTHROPIC_API_KEY=sk-ant-...
```

> Using another provider? Swap the model string below, e.g. `"openai:gpt-4.1"`
> with `OPENAI_API_KEY` in `.env`. Everything else stays identical: that is the
> point of the harness.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

MODEL = "anthropic:claude-haiku-4-5"   # swap provider here if needed
llm = init_chat_model(MODEL)

llm.invoke("Say 'ready' and nothing else.").content

### Restore yesterday's world

Every day-3 notebook starts from the same booking agent you built on day 2.

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool

HOTELS = {"Hotel Anders":  {"free": True,  "eur": 92},
          "Pension Krumm": {"free": True,  "eur": 61},
          "Grand Mitte":   {"free": False, "eur": 210}}

@tool
def check_availability(hotel: str) -> dict:
    """Live availability and price (EUR/night) for one hotel."""
    return HOTELS.get(hotel, {"error": f"unknown hotel: {hotel}"})

@tool
def list_hotels() -> list:
    """All hotels we can book, cheapest first."""
    return sorted(HOTELS, key=lambda h: HOTELS[h]["eur"])

@tool
def book_room(hotel: str, nights: int) -> dict:
    """Book a room. THIS ONE WRITES."""
    return {"confirmation": f"GIU-{abs(hash(hotel + str(nights))) % 10000:04d}"}

INSTRUCTIONS = ("You are the GIU booking assistant. "
                "Check availability before booking. Prices in EUR.")
print("world restored")

## 1 · The checkpointer: something that writes the list down

Day 2's lesson was *the agent remembers nothing, the list does*. You proved it:
a fresh `invoke` had no idea who you were.

A **checkpointer** is just a place where that list gets saved. After every step
of the loop it writes the messages down, filed under a name you choose. On the
next call it reads them back before the model sees anything.

Two new things in the code below:

- `checkpointer=InMemorySaver()` gives the agent somewhere to write
- `thread_id` is the name of one conversation. Think of it as one chat window.
  Same name, same history. A different name, a completely separate chat.

Notice what you do **not** do any more: you never pass the old messages back.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(model=llm,
    tools=[check_availability, list_hotels, book_room],
    system_prompt=INSTRUCTIONS,
    checkpointer=InMemorySaver())        # ← the new argument

amr  = {"configurable": {"thread_id": "amr"}}
sara = {"configurable": {"thread_id": "sara"}}

agent.invoke({"messages": [HumanMessage("I need a room in Berlin, 2 nights.")]}, amr)
out = agent.invoke({"messages": [HumanMessage("Make it three nights.")]}, amr)
print("thread amr :", out["messages"][-1].content[:150])

out = agent.invoke({"messages": [HumanMessage("How many nights did I ask for?")]}, sara)
print("thread sara:", out["messages"][-1].content[:150])   # clean slate, different thread

Read those two answers. On thread `amr` it knew the booking had changed to three
nights. On thread `sara` it had no idea, because that is a different chat.

In a real app your backend hands out the ids: one per user session, or one per
support ticket. The agent itself never knows or cares.

In [ ]:
# the whole transcript is inspectable, it's data, not vibes
state = agent.get_state(amr)
for m in state.values["messages"]:
    print(m.type.upper().ljust(6), "→", str(m.content)[:80])

### Where is that list actually stored?

`InMemorySaver` keeps it in your program's memory. In a notebook, *your program
is the Jupyter kernel*, and the kernel stays alive between cells. That is why the
cells above worked: the same kernel, the same memory.

Press **Restart Kernel** and it is all gone. Same as closing a program.

`SqliteSaver` writes to a real file on disk instead (`checkpoints.db`). Nothing
else in your code changes. Now the conversation outlives the kernel, the laptop
and the deploy. Swap SQLite for Postgres later and it is a production system.

| | lives in | survives a restart? | good for |
|---|---|---|---|
| `InMemorySaver` | the kernel's memory | no | notebooks, tests, demos |
| `SqliteSaver` | a file on disk | yes | anything real |

In [ ]:
# swap ONE line and conversations survive a restart
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

saver = SqliteSaver(sqlite3.connect("checkpoints.db", check_same_thread=False))

durable = create_agent(model=llm,
    tools=[check_availability, list_hotels, book_room],
    system_prompt=INSTRUCTIONS,
    checkpointer=saver)

durable.invoke({"messages": [HumanMessage("Hold a room at Pension Krumm for me.")]},
               {"configurable": {"thread_id": "amr-durable"}})
print("saved to checkpoints.db")

**Your turn (1): prove it.**

1. **Kernel → Restart Kernel.** Everything in memory is now gone.
2. Re-run only the setup cells and the SQLite cell above.
3. Ask *"what did I ask you to hold?"* on thread `amr-durable`.

It still knows. The program died and the conversation did not, because it was
never in the program to begin with. It was in a file.

Then try the same thing with the `InMemorySaver` agent and watch it fail. That
contrast is the whole lesson.

In [ ]:
# after the restart, prove persistence here


## 2 · Now it can wait for you

Here is the payoff. Because the run can be saved and picked up again, it can
also **stop in the middle** and wait for a person.

`HumanInTheLoopMiddleware` takes a list of tool names. When the model asks for
one of those tools, the harness does not run it. It saves the half-finished run
and hands control back to you.

Reads are not in the list, so `check_availability` keeps working normally. Only
`book_room` waits.

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

guarded = create_agent(model=llm,
    tools=[check_availability, list_hotels, book_room],
    system_prompt=INSTRUCTIONS,
    checkpointer=InMemorySaver(),        # pausing REQUIRES persistence
    middleware=[HumanInTheLoopMiddleware(
        interrupt_on={"book_room": True})])   # reads flow, this one waits

cfg = {"configurable": {"thread_id": "guarded-1"}}
out = guarded.invoke({"messages": [HumanMessage(
    "Book me the cheapest available room for 2 nights.")]}, cfg)

state = guarded.get_state(cfg)
print("run is parked before:", state.next)
print("\nwhat it wants to do:")
print(state.tasks[0].interrupts if state.tasks else "no interrupt")

Two things to notice.

`state.next` tells you the run is **not finished**. It is parked, mid-loop.

The interrupt payload is what you would show a human in a real interface: *"the
agent wants to call `book_room(hotel=..., nights=2)`. Approve?"* A web app would
render that as a button. Here we approve it in the next cell.

And the run can sit there for a week. It is a row in the checkpointer, not a
program stuck waiting in a loop.

In [ ]:
# a human signs off, the run reloads and continues exactly where it stopped
from langgraph.types import Command

resumed = guarded.invoke(Command(resume=[{"type": "approve"}]), cfg)
print(resumed["messages"][-1].content)

### The four answers you can give

`resume` takes a **list** of decisions, one per interrupted tool call, because a
single lap can propose several at once. There are four kinds:

| decision | does the tool run? | payload |
|---|---|---|
| **approve** | yes, exactly as proposed | `{"type": "approve"}` |
| **edit** | yes, with *your* arguments | `{"type": "edit", "edited_action": {"name": "book_room", "args": {...}}}` |
| **reject** | no. The model is told, and may try something else | `{"type": "reject", "message": "too expensive"}` |
| **respond** | no. You answer the model yourself | `{"type": "respond", "message": "ask the student first"}` |

Look again at that last `invoke`: it carries **no messages at all**. The
`thread_id` inside `cfg` is the only thing linking it to the parked run.

You can also limit what is on offer. Instead of `True`, pass a config:

```python
interrupt_on={"book_room": {"allowed_decisions": ["approve", "reject"]}}
```

Now nobody can quietly edit the arguments on their way through.

**Your turn (2): try the other three.** Use a **fresh `thread_id` each time**,
because one parked run can only be answered once.

1. **reject** with a message. What does the agent say next: does it give up, or
   propose something different?
2. **edit** the booking down to 1 night. Check the confirmation that comes back
   uses *your* number, not the model's.
3. **respond** with `"the student has not paid yet"`. The tool never runs, but
   the model still receives your sentence and reacts to it.

In [ ]:
# your reject / edit / respond experiments here


**Your turn (3)**, over-guard it: add `check_availability` to `interrupt_on`
and run a normal question. Count how many approvals one simple request now
costs you.

Guardrails have a price. Put them where writes happen, **not everywhere.**

In [ ]:
# your over-guarded agent here


---
## Wrap

`checkpointer=` + `thread_id` → conversations that continue and survive restarts ·
`HumanInTheLoopMiddleware` → the run parks before a write and resumes with
`Command(resume=…)` · the pause lives in the checkpointer, which is why the two
belong in the same session.

**Next (3c):** the system prompt is one big string. Time to give the agent a
binder instead.